# Practical 2 — RAG App (Fixed)

## Ingestion Flow + Query Flow

This notebook fixes the retrieval problems in the original **Practical 2** RAG pipeline so it consistently answers from the right section of the source page, using the right context.

### What was actually wrong

Several issues were causing wrong or inconsistent answers:

1. **Non-idempotent ingestion.** `vector_db.add_documents(...)` was called against a Chroma collection with no reset logic. Re-running cells while iterating (very common in a notebook) silently appends duplicate chunks to the same collection every time. Duplicated chunks skew the top‑k results, so the answer you get depends on how many times you already ran the notebook — not just on the question.
2. **Chunk size too small for the content's structure.** The page has a bulleted "**Here's the key info, at a glance:**" summary with about 7 items. At `chunk_size=500`, that single list gets sliced into 8–9 separate chunks. With the retriever capped at `k=6`, it is *structurally impossible* to retrieve the whole list — some items are dropped before the LLM ever sees them, no matter how good the embeddings or the LLM are.
3. **No text cleaning.** `WebBaseLoader` pulls in bylines, irregular whitespace and other page furniture along with the article body. That noise gets embedded and chunked right alongside the real content, and can crowd out genuinely relevant chunks in the top‑k results.
4. **Unspecified similarity space.** The embedding model wasn't told to normalize its vectors, and Chroma defaulted to squared‑L2 distance instead of cosine — a common source of subtly inconsistent similarity rankings with sentence‑transformer‑style models like this one.
5. **Vector similarity is the wrong tool for "give me everything under this exact heading" questions.** The "at a glance" list is short and mixes several topics in a few words each, while the article separately elaborates on each of those topics later on, in longer, more topically-focused sections. Those expanded sections match many natural-language queries *better* than the terse summary chunk does — so tuning `k`, `lambda_mult`, or the similarity metric never reliably surfaces the actual list. This is a positional/structural request, not a semantic-similarity one, discovered empirically while testing (see steps 15 and 23–24 below).

### What this notebook fixes

- Ingestion is made idempotent: a fresh, explicit in‑memory Chroma collection every run, reset if it already exists
- Chunk size/overlap increased so structured lists mostly survive splitting intact
- Loaded text is cleaned (whitespace collapsed) before chunking
- Embeddings are normalized and Chroma is set to cosine similarity to match
- Retrieval tuned to plain, relevance-ranked similarity search with a larger `k` (an earlier attempt at MMR here made completeness *worse* — see step 15's note)
- A **hybrid, heading-anchored retriever** added for questions tied to a specific heading: it finds that heading by exact text match and pulls in the chunks that follow it in document order, merged with the normal vector search results
- The prompt is nudged to enumerate *every* item found in context, not just the first few

### Ingestion flow
Web page → Load → **Clean** → Chunk (bigger) → Embed (**normalized**) → Chroma (**cosine, reset each run**) → Retriever (**similarity + heading-anchored hybrid**)

### Query flow
Question → Retriever → Relevant chunks → Prompt (list‑complete) → Cohere → Answer

**Important:** the final questions use `rag_chain.invoke()` (or `hybrid_rag_chain.invoke()` for heading-anchored questions) so the retriever selects the context automatically.


## 1. Install dependencies

In [ ]:
!pip install -q -U \
langchain \
langchain-classic \
langchain-community \
langchain-text-splitters \
langchain-chroma \
langchain-huggingface \
langchain-cohere \
chromadb \
transformers \
sentence-transformers \
beautifulsoup4

## 2. Imports

In [ ]:
import os
import re
from getpass import getpass

import chromadb
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_cohere import ChatCohere
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

## 3. Set a User-Agent for polite scraping

`WebBaseLoader` sends a plain HTTP request. Without a `USER_AGENT`, `langchain_community` prints a warning and some sites treat the request less predictably. Setting one is harmless and removes that noise.

In [ ]:
os.environ.setdefault("USER_AGENT", "Practical2-RAG-Notebook/1.0")

## 4. Set Cohere API key

In [ ]:
os.environ["COHERE_API_KEY"] = getpass("Enter your Cohere API key: ")

## 5. Initialize the LLM

In [ ]:
llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0,
    max_tokens=2000
)

# INGESTION FLOW

## 6. Load the IBM webpage

In [ ]:
url = "https://www.ibm.com/new/announcements/ibm-granite-3-1-powerful-performance-long-context-and-more"

doc = WebBaseLoader(url).load()

print("Number of documents loaded:", len(doc))

## 7. Inspect the loaded text

In [ ]:
print(doc[0].page_content[:1500])

## 8. Clean the loaded text (fix #3: noisy scrape)

`WebBaseLoader` returns raw visible text, including irregular blank lines and page furniture (bylines, etc.) mixed in around the article body. We can't reliably strip *all* boilerplate without inspecting the raw HTML tag by tag, but the cheap, safe, universally-useful fix is to collapse the runs of whitespace it leaves behind. That keeps each chunk dense with actual content instead of padding, so more real text — and less noise — fits inside every `chunk_size` window downstream.

In [ ]:
def clean_text(text: str) -> str:
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

for d in doc:
    d.page_content = clean_text(d.page_content)

print(doc[0].page_content[:1500])

## 9. Chunk the document (fix #2: chunk size vs. content structure)

The original settings (`chunk_size=500`, `chunk_overlap=80`) chop the "**Here's the key info, at a glance:**" bullet list — about 7 items, several of them 200–400+ characters long — into roughly 8–9 separate chunks. The retriever is capped at `k=6`, so even a *perfect* retriever cannot return the whole list: part of it is always missing before the LLM sees anything. This is exactly the query that fails at the end of the notebook ("list all items following ... at a glance").

We increase `chunk_size` so the list mostly survives in one or two chunks, and scale the overlap proportionally so we don't lose context at chunk boundaries:

- `chunk_size = 1200`
- `chunk_overlap = 200`

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200
)

recursive_chunks = text_splitter.split_documents(doc)

print("Number of chunks:", len(recursive_chunks))

## 10. Inspect the first 5 chunks

In [ ]:
for i, chunk in enumerate(recursive_chunks[:5], start=1):
    print(f"\n--- CHUNK {i} ---")
    print(chunk.page_content)
    print("Metadata:", chunk.metadata)

## 11. Optional inspection around "Here's the key info, at a glance:"

This is only for understanding the chunking result. It is **not** used to manually supply context to the final RAG query. With the larger chunk size, the whole list should now fit into far fewer chunks than before — verify that here.

In [ ]:
for index, chunk in enumerate(recursive_chunks):
    if "at a glance" in chunk.page_content.casefold():
        for nearby_index in range(
            max(0, index - 1),
            min(len(recursive_chunks), index + 4),
        ):
            print(f"\n--- CHUNK {nearby_index} ---")
            print(recursive_chunks[nearby_index].page_content)
        break
else:
    print("Heading not found in the chunks.")

## 12. Create the embedding model (fix #4: similarity space)

We explicitly normalize the embedding vectors. Sentence-transformer-style models (like this Granite embedding model) are trained and evaluated with cosine similarity in mind. Normalizing the vectors here and pairing that with a cosine similarity space in Chroma (next step) gives standard, consistent similarity rankings instead of relying on Chroma's default squared-L2 distance over un-normalized vectors.

In [ ]:
embeddings_model = HuggingFaceEmbeddings(
    model_name="ibm-granite/granite-embedding-30m-english",
    encode_kwargs={"normalize_embeddings": True}
)

## 13. Create the Chroma vector store (fix #1: idempotent ingestion)

The original code created a `Chroma(...)` instance with no explicit client and no reset logic. Depending on the Chroma version and working directory, this can silently reuse a collection left over from a previous run — and even within a single session, re-running the "add documents" cell appends duplicates to the same collection. Duplicated chunks skew the top-k results and make answers change between runs for no visible reason.

Here we:
- Use an explicit **in-memory client** (`chromadb.EphemeralClient()`), so nothing persists between kernel runs by accident.
- **Delete any existing collection with the same name** first, so re-running this cell — or the whole notebook — always starts from a clean slate.
- Set the collection's similarity space to **cosine**, matching the normalized embeddings above.

In [ ]:
chroma_client = chromadb.EphemeralClient()

# Idempotency: always start this collection from a clean slate, even on reruns
try:
    chroma_client.delete_collection("raw_web_recursive_test")
except Exception:
    pass

vector_db = Chroma(
    client=chroma_client,
    collection_name="raw_web_recursive_test",
    embedding_function=embeddings_model,
    collection_metadata={"hnsw:space": "cosine"}
)

## 14. Add the chunks to the vector store

Because step 13 resets the collection first, "chunks added" and "chunks actually in the collection" below should always match exactly. If they ever don't, that's a sign something is stacking duplicates again.

In [ ]:
vector_db.add_documents(recursive_chunks)

print("Chunks added to Chroma:", len(recursive_chunks))
print("Chunks actually in the collection:", vector_db._collection.count())

## 15. Create the retriever (fix #2 continued: retrieval strategy)

`k=6` alone wasn't the real problem — it was `k=6` combined with 500-character chunks that made the "at a glance" list impossible to retrieve in full. Now that chunks are larger, far fewer chunks are needed to cover the whole list.

**A note on MMR (Maximal Marginal Relevance):** an earlier version of this notebook used `search_type="mmr"` here, reasoning that it would avoid returning several near-identical, overlapping chunks. In practice this backfired for exhaustive "list everything in this section" questions: MMR intentionally penalizes selecting a chunk that's *similar* to one it already picked, to maximize diversity. The bullets under "Here's the key info, at a glance:" are all part of the same announcement and topically related to each other — so once MMR picked one bullet-chunk, it actively steered away from the next ones in favor of unrelated parts of the page, handing the LLM an incomplete list and producing "I don't know" (correctly, per the prompt's "don't guess" instruction — the retrieval was the actual problem).

The lesson: diversity is the wrong objective when the question needs *everything in one coherent region* of the document, not a broad sample across it. Plain relevance-ranked similarity search, with a slightly larger `k`, is the more reliable default here — and it's cheap now that chunks are bigger.

In [ ]:
retriever = vector_db.as_retriever(
    search_kwargs={"k": 10}
)

## 16. Test retrieval by itself

This lets us inspect what the retriever selects **before** the LLM generates an answer.

In [ ]:
query = "What is the name of the main model being discussed? Be very specific."

retrieved_docs = retriever.invoke(query)

for i, retrieved_doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- RETRIEVED CHUNK {i} ---")
    print(retrieved_doc.page_content)

# QUERY FLOW

## 17. Define the prompt

We keep the "answer only from context / say I don't know" instruction, and add one line asking the model to enumerate every item it finds when a question asks for a list. This is a small but important belt-and-suspenders fix: even once the retriever *does* hand over the complete list (thanks to the fixes above), nothing stopped the LLM itself from summarizing or truncating it — the original prompt never told it not to.

In [ ]:
prompt = ChatPromptTemplate.from_template(
    """
Answer the question using only the context provided below.

If the answer cannot be found in the context, say "I don't know."
Do not make up an answer.
If the question asks for a list, include every item that appears
in the context word for word in meaning - do not omit, merge, or summarize any of them.

Context:
{context}

Question:
{input}

Answer:
"""
)

## 18. Build the full RAG chain

In [ ]:
combine_docs_chain = create_stuff_documents_chain(
    llm,
    prompt
)

rag_chain = create_retrieval_chain(
    retriever,
    combine_docs_chain
)

## 19. Run the full RAG query

This is the important correction carried over from the original notebook.

We use:

```python
rag_chain.invoke(...)
```

instead of manually passing:

```python
recursive_chunks[3:9]
```

The retriever chooses the context automatically — and now that ingestion is clean, chunking respects the page's structure, and retrieval uses relevance-ranked similarity search over a normalized/cosine space, the context it chooses is far more reliable than the original.

In [ ]:
question = "What is the name of the main model being discussed? Be very specific."

result = rag_chain.invoke({
    "input": question
})

print(result["answer"])

## 20. Inspect the chunks actually used for the answer

In [ ]:
for i, retrieved_doc in enumerate(result["context"], start=1):
    print(f"\n--- RAG CONTEXT {i} ---")
    print(retrieved_doc.page_content)

## 21. Second RAG query

This is the query that exposed the original bug most clearly: it needs *every* bullet under "Here's the key info, at a glance:". Even after fixing ingestion, chunk size, cleaning, normalization and retrieval tuning, this specific query can still come back as "I don't know" or incomplete — run the debug cell below with this exact question if that happens, then see steps 23–25 for why and how to fix it.

In [ ]:
question = (
    'List all the items immediately following '
    '"Here\'s the key info, at a glance:" in the article.'
)

result = rag_chain.invoke({
    "input": question
})

print(result["answer"])

## 22. Debugging retrieval for a specific question

Whenever an answer looks wrong or incomplete, check retrieval *before* suspecting the LLM or the prompt — `rag_chain.invoke()` hides this step. This is the fastest way to tell "the retriever didn't find the right chunks" apart from "the LLM had the right chunks and still got it wrong." Reuse this cell for any question that misbehaves.

In [ ]:
debug_question = question  # swap in whichever question is misbehaving

debug_chunks = retriever.invoke(debug_question)

print(f"Retrieved {len(debug_chunks)} chunks for: {debug_question!r}\n")
for i, c in enumerate(debug_chunks, start=1):
    print(f"--- CHUNK {i} ---")
    print(c.page_content[:300])
    print()

## 23. Why plain similarity search still misses this section

Running the debug cell above for the "list items" question reveals the real story: **none of the top-10 retrieved chunks contain "Here's the key info, at a glance:" or its bullets at all.** Instead you get the newsletter/footnotes footer, a breadcrumb/header chunk, and several *expanded* body sections — "Expanded context length," "Granite Guardian 3.1: detecting hallucinations...," "Document deciphering and agentic AI," the embeddings section — each of which elaborates on one bullet's topic later in the article, in more focused language than the terse combined bullet list.

That's not a `k` or `lambda_mult` problem — it's a structural mismatch between what embeddings are good at and what this question needs:

- The "at a glance" chunk mixes several different topics (context length, embeddings, guardrails, licensing, open-source tooling) in a few words each. Averaged over a chunk, that heterogeneous content doesn't embed *strongly* toward any single topic.
- The later expanded sections are each narrowly about one topic, so they score *more* strongly against almost any specific query about that topic than the terse summary chunk does — even though the summary chunk is the one actually being asked about.
- Generic words in the query ("list," "items," "article") give weak but nonzero similarity to generic boilerplate (nav breadcrumbs, footer, newsletter signup), which is enough to nudge that boilerplate into a top-10 slot when nothing else stands out strongly.

No amount of tuning `k`, `lambda_mult`, or the similarity metric fixes this, because the request itself isn't "find the passage most semantically similar to my query" — it's "find this exact heading, then give me everything that comes right after it." That's a positional/structural request, not a semantic one, and vector search alone was never going to reliably answer it.

## 24. A hybrid retriever: anchor on the heading, then expand sequentially

Instead of asking embeddings to do a job they're not suited for, we detect when a question references a specific heading/phrase, locate that chunk directly by exact text match (not similarity), and pull in the chunks that immediately follow it in document order — exactly what the manual inspection cell (step 11) already does for debugging, just wired into retrieval itself. The normal vector retriever still runs and gets merged in, so this keeps working for questions that aren't anchored to a specific heading too.

In [ ]:
from langchain_core.runnables import RunnableLambda

ANCHOR_PHRASES = ["at a glance"]  # extend this for other headings you care about
CHUNKS_AFTER_ANCHOR = 6

def hybrid_retrieve(query: str):
    query_lower = query.lower()

    anchored_hits = []
    for phrase in ANCHOR_PHRASES:
        if phrase in query_lower:
            for i, chunk in enumerate(recursive_chunks):
                if phrase in chunk.page_content.lower():
                    start = max(0, i - 1)
                    end = min(len(recursive_chunks), i + CHUNKS_AFTER_ANCHOR)
                    anchored_hits = recursive_chunks[start:end]
                    break
            break

    vector_hits = retriever.invoke(query)

    seen = {c.page_content for c in anchored_hits}
    combined = anchored_hits + [d for d in vector_hits if d.page_content not in seen]
    return combined

hybrid_retriever = RunnableLambda(hybrid_retrieve)

hybrid_rag_chain = create_retrieval_chain(
    hybrid_retriever,
    combine_docs_chain
)

## 25. Second RAG query, take 2 (hybrid retrieval)

Same question, this time routed through `hybrid_rag_chain` instead of `rag_chain`.

In [ ]:
question = (
    'List all the items immediately following '
    '"Here\'s the key info, at a glance:" in the article.'
)

result = hybrid_rag_chain.invoke({
    "input": question
})

print(result["answer"])
print("\nChunks used:", len(result["context"]))

## What changed from the original notebook, and why answers are now consistent

| Problem | Fix | Why it matters |
|---|---|---|
| Chroma collection re-used/duplicated on reruns | Explicit `EphemeralClient`, collection deleted and recreated each run | Retrieval no longer depends on how many times you've already run the notebook |
| `chunk_size=500` split the "at a glance" list across 8–9 chunks, more than `k=6` could ever return | `chunk_size=1200`, `chunk_overlap=200` | The list now mostly fits in 1–2 chunks, so it's actually retrievable in full |
| Scraped text had uneven whitespace/boilerplate mixed with the article | Regex whitespace cleanup before chunking | Chunks are denser with real content, less noise competing in the top-k |
| Un-normalized embeddings + Chroma's default L2 distance | `normalize_embeddings=True` + `hnsw:space: cosine` | Standard, consistent similarity rankings for this embedding model |
| Prompt didn't ask the model to be exhaustive | Added an explicit "include every item" instruction | Removes the last place an answer could still get silently truncated |
| *(regression, caught during testing)* MMR retrieval traded relevance for diversity, and actively avoided returning multiple chunks from the same topically-related list — producing incomplete context and "I don't know" | Reverted to plain similarity search with a larger `k=10` | Diversity is the wrong objective when a question needs *everything in one coherent region* of the document, not a broad sample across it |
| *(caught during further testing)* Even plain similarity search never ranked the actual "at a glance" chunk in the top-10 — the article's expanded, topic-focused sections outscored the terse, multi-topic summary chunk for most natural-language queries | Added a **hybrid, heading-anchored retriever**: find the heading by exact text match, then take the chunks that follow it in document order, merged with the normal vector results | "Give me everything under this exact heading" is a structural/positional request, not a semantic-similarity one — no embedding tuning fixes that mismatch |

The retriever is still created only once, `recursive_chunks[3:9]` is never manually passed to the LLM, and final questions still go through `rag_chain.invoke()` (or `hybrid_rag_chain.invoke()`) — that part of the original redesign was correct. The real query flow is now:

**Question → Retriever (similarity, cosine — or heading-anchored hybrid when relevant) → Complete, non-redundant chunks → Prompt (exhaustive) → Cohere → Answer**

### Further improvements, if you want to go further
- For true idempotency *across kernel restarts* (not just within one run), persist Chroma to disk and give each chunk a deterministic `id` (e.g. a hash of its content) so re-ingesting the same page upserts instead of duplicating.
- For pages with much longer structured sections, a **parent-document / hierarchical retriever** (retrieve a small chunk, then expand to its parent section) scales better than hand-picking anchor phrases per page, and generalizes the trick used in step 24.
- The `ANCHOR_PHRASES` list in step 24 is manual and page-specific. A more general version would detect "list everything under heading X" phrasing automatically (e.g. by extracting quoted text from the question) instead of hardcoding known headings.
